# Synthetic Gaze with the Statistical Generator

The statistical generator simulates eye movement velocities from parametrised distributions,
implementing the simulator described in section 3 of:

> Fuhl, W., Santini, T., Kuebler, T., Castner, N., Rosenstiel, W., & Kasneci, E. (2018).
> *Eye movement simulation and detector creation to reduce laborious parameter adjustments.*
> arXiv

It builds a sequence of fixations, saccades and smooth pursuits, gives each one a velocity
profile, and interpolates the result to the sampling rate you ask for.

It emits a **velocity magnitude**, not a position: the underlying model has no directions, so
nothing here converts the signal into gaze coordinates.

The generation interface is still **experimental**: names and behaviour may change without
that counting as a breaking change, so pin the pymovements version if you depend on it. The
framework warns about this on every call, which the next cell silences for readability.

This notebook shows how to:

1. Generate a sequence and inspect it
2. Plot the velocity profiles
3. Condition the generator on an existing fixation sequence
4. Request different sampling rates
5. Reproduce a run from its metadata
6. Condition on fixations detected in a real recording

In [ ]:
import dataclasses
import warnings

import numpy as np
import polars as pl

import pymovements as pm
from pymovements.synthetic.generators import StatisticalConfig

# Acknowledged in the introduction above; silenced here so it does not repeat in every output.
warnings.filterwarnings('ignore', category=pm.ExperimentalWarning)

## 1. Generate a Sequence

The configuration holds everything: the target sampling rate, how long a sequence to produce,
the seed, and the parameters of each event type. Here we keep the defaults and only ask for two
seconds at 1000 Hz, with a little measurement noise.

In [ ]:
config = StatisticalConfig(
    sampling_rate=1000.0,
    duration_s=2.0,
    seed=42,
    noise_std_dps=1.5,
)

gaze = pm.synthetic.generate('statistical', config)

print('samples:', gaze.samples.shape)
print('columns:', gaze.samples.columns)

gaze.events.frame

The generator reports which event each part of the signal belongs to, so the output carries its
own ground truth. Event types are drawn uniformly unless you prescribe a sequence.

In [ ]:
# A prescribed sequence, repeated until the duration is filled.
alternating = pm.synthetic.generate(
    'statistical',
    dataclasses.replace(config, event_sequence=('fixation', 'saccade')),
)

alternating.events.frame['name'].value_counts()

## 2. Plot the Velocity Profiles

`tsplot` draws any numeric column against the `time` column, and `show_events=True` shades the
segments. Saccades are the Gamma-shaped bursts, smooth pursuits rise through a sigmoid onset and
then hold, and fixations stay near zero apart from their jitter.

Note that `traceplot` does *not* apply here: it needs gaze coordinates, and this generator
produces velocity magnitudes with no direction, so there is no path to trace.

In [ ]:
pm.plotting.tsplot(
    gaze,
    'velocity_magnitude',
    show_events=True,
    title='statistical, 2 s at 1000 Hz',
)

## 3. Condition on a Fixation Sequence

Instead of drawing the sequence, you can hand the generator one. The type and duration of every
segment then comes from the input. Fixation sequences have gaps where their saccades were, and
those gaps are filled with saccades so the output covers the same time span as the input.

In [ ]:
fixations = pm.Events(
    name=['fixation'] * 4,
    onsets=[0, 280, 610, 870],
    offsets=[240, 550, 820, 1150],
    time_unit='ms',
)

conditioned = pm.synthetic.generate(
    'statistical',
    StatisticalConfig(sampling_rate=1000.0, seed=42),
    events=fixations,
)

print('samples:', conditioned.samples.height)

conditioned.events.frame

## 4. Request a Different Sampling Rate

The profiles are built at a much higher internal rate and interpolated down, so any target rate
is served exactly.

In [ ]:
rows = []
for rate in (250.0, 500.0, 1000.0, 2000.0):
    rate_config = dataclasses.replace(config, sampling_rate=rate)
    produced = pm.synthetic.generate('statistical', rate_config)
    rows.append({'rate [Hz]': rate, 'samples': produced.samples.height})

pl.DataFrame(rows)

## 5. Reproduce a Run

Every run records its configuration and the seed it used, including when the seed was drawn for
you. Rebuilding the configuration from the metadata reproduces the signal exactly.

In [ ]:
unseeded = pm.synthetic.generate(
    'statistical',
    StatisticalConfig(sampling_rate=1000.0, duration_s=1.0),
)

recorded = StatisticalConfig.from_dict(unseeded.metadata['synthetic']['config'])
repeated = pm.synthetic.generate('statistical', recorded)

print('drawn seed:', recorded.seed)
print('reproduced:', repeated.samples.equals(unseeded.samples))

## 6. Visualize a Conditioned Run

The same plot for the conditioned sequence from step 3. The four fixations we supplied are there
at their original onsets, with the saccades that were inserted into the gaps between them.

In [ ]:
pm.plotting.tsplot(
    conditioned,
    'velocity_magnitude',
    show_events=True,
    title='conditioned on four fixations',
)

## 7. Condition on Fixations from a Real Recording

The conditioning input does not have to be handmade. Detect fixations in a recording and the
generator will follow them, which is the case it is really meant for: real event timing, synthetic
samples.

In [ ]:
dataset = pm.Dataset('ToyDataset', path='data/ToyDataset')
dataset.download()
dataset.load()

recording = dataset.gaze[0]
recording.pix2deg()
recording.pos2vel()
recording.detect('ivt')

# A short stretch is enough to see the structure.
detected = pm.Events(
    recording.events.frame.filter(pl.col('name') == 'fixation').head(8),
)

print('detected fixations:', len(detected))
detected.frame.select('name', 'onset', 'offset', 'duration')

The gaps between those fixations are the recording's real saccades, which the detector did not
label. They are filled with synthetic saccades, so the output spans the same window as the input.

In [ ]:
from_recording = pm.synthetic.generate(
    'statistical',
    StatisticalConfig(sampling_rate=recording.experiment.sampling_rate, seed=42),
    events=detected,
)

pm.plotting.tsplot(
    from_recording,
    'velocity_magnitude',
    show_events=True,
    title='conditioned on fixations detected in ToyDataset',
)

### Compare it against the recording

Because the generator emits a velocity magnitude, the recording's own velocity magnitude is what
it can be compared against. This is where the defaults show their limits: they were never fitted
to any recording, so fixations come out too quiet and saccades too fast.

In [ ]:
def magnitude_summary(values, label):
    """Summarise a velocity magnitude in degrees per second."""
    return {
        'signal': label,
        'median': round(float(np.nanmedian(values)), 1),
        'p95': round(float(np.nanpercentile(values, 95)), 1),
    }


window = recording.samples.filter(
    pl.col('time').is_between(
        detected.frame['onset'].to_list()[0], detected.frame['offset'].to_list()[-1],
    ),
)
recorded = np.linalg.norm(np.vstack(window['velocity'].to_numpy()), axis=1)

# Narrow the ranges towards what the recording actually shows.
tuned = pm.synthetic.generate(
    'statistical',
    StatisticalConfig(
        sampling_rate=recording.experiment.sampling_rate,
        seed=42,
        fixation_consistency_min_dps=4.0,
        fixation_consistency_max_dps=11.0,
        saccade_peak_velocity_min_dps=80.0,
        saccade_peak_velocity_max_dps=260.0,
    ),
    events=detected,
)

pl.DataFrame([
    magnitude_summary(recorded, 'recording'),
    magnitude_summary(from_recording.samples['velocity_magnitude'], 'synthetic, defaults'),
    magnitude_summary(tuned.samples['velocity_magnitude'], 'synthetic, tuned'),
])